# Flickr30k DenseMap negative-result closeout

Khóa kết quả âm của `ExpansionNet v2 → MobileNetV3-Large` với
`Dense Mapped Logit-KD` trên ba seed. Notebook không huấn luyện lại,
không lượng tử hóa và không đọc Held-out Test.

## Kaggle input bắt buộc — đúng 1 input

1. Output `F30K_T1S1_DenseMap_Aggregate`.

Runtime: **CPU / Accelerator None**, Internet Off, dự kiến dưới 1 phút.
Notebook có thể chạy song song với `F30K_Global_Val_Lock`.

## Cell 1 — Verify aggregate and close the negative branch

In [1]:
EXPERIMENT_ID = "F30K_DenseMap_Closeout"


import csv
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_BASE = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working"))
WORK_ROOT = WORK_BASE / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def verify_manifest(root):
    root = Path(root)
    manifest_path = root / "MANIFEST.json"
    manifest = read_json(manifest_path)
    assert manifest["status"] == "complete"
    assert manifest["test_set_status"] == "sealed"
    assert manifest["test_set_used"] is False
    for artifact in manifest["artifacts"]:
        path = root / artifact["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(artifact["size_bytes"])
        assert sha256(path) == artifact["sha256"]
    return manifest_path, manifest

def write_manifest(paths, schema_version):
    artifacts = []
    for path in paths:
        path = Path(path)
        artifacts.append({
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })
    write_json(WORK_ROOT / "MANIFEST.json", {
        "schema_version": schema_version,
        "experiment_id": EXPERIMENT_ID,
        "status": "complete",
        "artifacts": artifacts,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

AGGREGATE_ID = "F30K_T1S1_DenseMap_Aggregate"
aggregate_path = unique_named(AGGREGATE_ID + "_RESULT.json")
aggregate = read_json(aggregate_path)
aggregate_manifest_path, _ = verify_manifest(aggregate_path.parent)

assert aggregate["status"] == "complete"
assert aggregate["experiment_id"] == AGGREGATE_ID
assert aggregate["pair_id"] == "T1S1_DenseMap"
assert aggregate["method"] == "Dense Mapped Logit-KD"
assert aggregate["seeds"] == [42, 123, 2026]
assert int(aggregate["run_count"]) == 3
assert int(aggregate["validation_images_per_run"]) == 1014
assert aggregate["selection_metric"] == "Validation CIDEr-D"
assert aggregate["consistent_positive_delta"] is False
assert aggregate["consistent_negative_delta"] is True
assert aggregate["decision"] == "STOP_VALIDATION_NO_GAIN"
assert aggregate["next_phase_authorized"] is None
assert aggregate["test_set_status"] == "sealed"
assert aggregate["test_set_used"] is False

combined = aggregate["combined_paired_bootstrap"]
assert int(combined["seed_count"]) == 3
assert int(combined["unique_validation_images"]) == 1014
assert int(combined["replicates"]) == 10000
assert float(combined["ci95_upper"]) < 0.0
assert float(combined["bootstrap_positive_probability"]) == 0.0
assert all(float(row["delta_CIDEr_D"]) < 0 for row in aggregate["runs"])

metric_names = ("BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D")
rows = []
for metric in metric_names:
    summary = aggregate["summary"][metric]
    rows.append({
        "metric": metric,
        "baseline_mean_x100": 100.0 * float(summary["baseline_mean"]),
        "kd_mean_x100": 100.0 * float(summary["kd_mean"]),
        "delta_mean_x100": 100.0 * float(summary["delta_mean"]),
        "delta_sample_sd_x100": 100.0 * float(summary["delta_sample_sd"]),
    })

closeout = {
    "schema_version": "f30k_t1s1_densemap_closeout_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "pair_id": "T1S1_DenseMap",
    "method": "Dense Mapped Logit-KD",
    "seeds": [42, 123, 2026],
    "metric_display_scale": 100,
    "validation_summary": rows,
    "combined_paired_bootstrap_CIDEr_D": combined,
    "scientific_conclusion": (
        "Dense Mapped Logit-KD reduced Validation quality consistently "
        "across all three seeds; the combined 95% paired-bootstrap interval "
        "for CIDEr-D is entirely below zero."
    ),
    "decision": "STOP_VALIDATION_NO_GAIN",
    "retraining_authorized": False,
    "quantization_authorized": False,
    "held_out_test_authorized": False,
    "retain_as_negative_result": True,
    "next_phase_authorized": None,
    "source_aggregate_result_sha256": sha256(aggregate_path),
    "source_aggregate_manifest_sha256": sha256(aggregate_manifest_path),
    "test_set_status": "sealed",
    "test_set_used": False,
}

result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
csv_path = WORK_ROOT / "validation_metrics_x100.csv"
report_path = WORK_ROOT / "REPORT.md"
write_json(result_path, closeout)
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

report_path.write_text(
    "# Flickr30k T1→S1 Dense Mapped Logit-KD closeout\n\n"
    f"- Mean Validation CIDEr-D: "
    f"`{100.0 * aggregate['summary']['CIDEr_D']['kd_mean']:.2f}`.\n"
    f"- Mean Δ CIDEr-D: "
    f"`{100.0 * aggregate['summary']['CIDEr_D']['delta_mean']:+.2f}`.\n"
    f"- Combined CI95: "
    f"`[{100.0 * float(combined['ci95_lower']):+.2f}; "
    f"{100.0 * float(combined['ci95_upper']):+.2f}]`.\n"
    "- Decision: `STOP_VALIDATION_NO_GAIN`.\n"
    "- Quantization/Test: `NOT AUTHORIZED`.\n"
    "- Held-out Test: `SEALED / NOT USED`.\n",
    encoding="utf-8",
)
write_manifest(
    [result_path, csv_path, report_path],
    "f30k_t1s1_densemap_closeout_manifest_v1",
)

print("Status                 : COMPLETE")
print("Branch                 : T1->S1 Dense Mapped Logit-KD")
print("Mean CIDEr-D x100     :",
      round(100.0 * aggregate["summary"]["CIDEr_D"]["kd_mean"], 2))
print("Mean delta x100       :",
      round(100.0 * aggregate["summary"]["CIDEr_D"]["delta_mean"], 2))
print("Combined CI95 x100    :",
      round(100.0 * float(combined["ci95_lower"]), 2),
      round(100.0 * float(combined["ci95_upper"]), 2))
print("Decision               : STOP_VALIDATION_NO_GAIN")
print("Held-out Test          : SEALED / NOT USED")

Status                 : COMPLETE
Branch                 : T1->S1 Dense Mapped Logit-KD
Mean CIDEr-D x100     : 37.17
Mean delta x100       : -10.38
Combined CI95 x100    : -12.7 -8.09
Decision               : STOP_VALIDATION_NO_GAIN
Held-out Test          : SEALED / NOT USED
